<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10 lab: convolutional networks on MNIST {.unnumbered}

This lab builds convolutional networks yourself and tests the ideas from
the Day 10 page:

- why a convolutional layer needs far fewer weights than a dense one;
- **padding**, and how the image shrinks through a stack of convolutions;
- a **VGG block**, and what happens when digits appear at positions the
  network never saw during training (translation);
- a **ResNet block** with a skip connection;
- what happens when test digits are **rotated**, with and without
  rotation augmentation.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**Runtime:** a few minutes on Colab's free CPU (faster with a GPU:
Runtime → Change runtime type). Keep `SEED = 0` so that your numbers
match the lab quiz. Training still varies a little between machines, so
the quiz accepts a range, and you upload your notebook at the end.

In [ ]:
import os, tempfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))
device = "cuda" if torch.cuda.is_available() else "cpu"

# MNIST is downloaded to a temporary directory, not next to this notebook
DATA_DIR = os.path.join(tempfile.gettempdir(), "kb8029_mnist")

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

## 1. The data (as in the Day 8 and 9 labs)

500 training and 100 validation images per digit from the official
training set, and 2,000 images from the official test set. Pixels are
standardized with statistics from the training images only. This time
the images keep their **2D shape**: a tensor of shape (images, 1 channel,
28, 28), because a convolutional layer works on the grid of pixels.

In [ ]:
mnist_train = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True)
mnist_test = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True)
X_all = mnist_train.data.numpy().astype(np.float32) / 255.0
y_all = mnist_train.targets.numpy()

rng = np.random.RandomState(SEED)
train_idx, val_idx = [], []
for digit in range(10):
    idx = rng.permutation(np.where(y_all == digit)[0])
    val_idx.append(idx[:100])
    train_idx.append(idx[100:600])
train_idx, val_idx = np.concatenate(train_idx), np.concatenate(val_idx)
test_idx = rng.choice(len(mnist_test), 2000, replace=False)

mu, sd = X_all[train_idx].mean(), X_all[train_idx].std()     # training images only
img = lambda X: torch.tensor((X - mu) / sd).unsqueeze(1)       # (N, 1, 28, 28)
X_train, X_val = img(X_all[train_idx]), img(X_all[val_idx])
X_test = img(mnist_test.data.numpy()[test_idx].astype(np.float32) / 255.0)
y_train, y_val = torch.tensor(y_all[train_idx]), torch.tensor(y_all[val_idx])
y_test = mnist_test.targets[test_idx]
print("train:", tuple(X_train.shape), " val:", tuple(X_val.shape), " test:", tuple(X_test.shape))

Two helpers used throughout: `count` gives a model's number of trainable
parameters, and `train` is the mini-batch training loop you wrote in the
Day 9 lab (Adam, cross-entropy), returning the validation accuracy after
the last epoch.

In [ ]:
count = lambda model: sum(p.numel() for p in model.parameters() if p.requires_grad)

def accuracy(model, X, y, batch_size=500):
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            correct += (model(X[i:i + batch_size].to(device)).argmax(1).cpu() == y[i:i + batch_size]).sum().item()
    return correct / len(X)

def train(model, X, y, epochs=5, lr=1e-3, batch_size=64, augment=None, seed=SEED):
    '''Mini-batch training with Adam and cross-entropy. `augment`, if given, is a
    function applied to every training batch (data augmentation).'''
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if augment is not None:
                xb = augment(xb)
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(xb.to(device)), yb.to(device))
            loss.backward()
            opt.step()
    return model

## 2. Dense vs. convolutional: counting weights

Build two models:

- **dense:** `Flatten`, `Linear(784, 10)`, `Tanh`, `Linear(10, 10)`;
- **convolutional:** `Conv2d(1, 10, kernel_size=5)` (10 filters of 5×5,
  stride 1, no padding), `Tanh`, `Flatten`, then a `Linear` layer to the
  10 outputs.

The convolution's output is 10 feature maps of 24×24 (a 5×5 kernel fits
24 times across 28 pixels without padding), so the final `Linear` layer
has 10 × 24 × 24 = 5,760 inputs.

In [ ]:
dense1 = nn.Sequential(nn.Flatten(), nn.Linear(784, 10), nn.Tanh(), nn.Linear(10, 10))
conv1 = todo("the convolutional model described above")
print("dense model:        ", count(dense1), "parameters")
print("convolutional model:", count(conv1), "parameters")
print("  of which the convolutional layer itself:", count(conv1[0]))

Now **duplicate the hidden layer** in both: a second `Linear(10, 10)` +
`Tanh` in the dense model, and a second `Conv2d(10, 10, kernel_size=5)` +
`Tanh` in the convolutional one. The second convolution shrinks the maps
to 20×20, so the final `Linear` layer now has 10 × 20 × 20 = 4,000 inputs.

In [ ]:
dense2 = nn.Sequential(nn.Flatten(), nn.Linear(784, 10), nn.Tanh(), nn.Linear(10, 10), nn.Tanh(),
                       nn.Linear(10, 10))
conv2 = todo("the convolutional model with two convolutional layers")
print(f"dense:         {count(dense1):6d} -> {count(dense2):6d} parameters (change {count(dense2) - count(dense1):+d})")
print(f"convolutional: {count(conv1):6d} -> {count(conv2):6d} parameters (change {count(conv2) - count(conv1):+d})")

Train the one-hidden-layer dense model and the one-layer convolutional
model for 5 epochs each and compare their validation accuracy.

In [ ]:
torch.manual_seed(SEED); dense_model = train(nn.Sequential(nn.Flatten(), nn.Linear(784, 10), nn.Tanh(), nn.Linear(10, 10)), X_train, y_train)
torch.manual_seed(SEED); conv_model = train(nn.Sequential(nn.Conv2d(1, 10, kernel_size=5), nn.Tanh(), nn.Flatten(),
                                                          nn.Linear(10 * 24 * 24, 10)), X_train, y_train)
acc_dense = accuracy(dense_model, X_val, y_val)
acc_conv = todo('validation accuracy of conv_model')
print(f"validation accuracy: dense {acc_dense:.3f}   convolutional {acc_conv:.3f}")

## 3. Padding, and how deep can you stack?

Without padding ("valid"), a 5×5 convolution with stride 1 shrinks each
side of the image by 4 pixels. Stack such layers on a 28×28 MNIST image
until the output would become empty, and count how many fit.

In [ ]:
x = torch.zeros(1, 1, 28, 28)
n_layers = 0
while True:
    layer = nn.Conv2d(x.shape[1], 8, kernel_size=5)
    if x.shape[-1] < 5:            # the 5x5 kernel no longer fits
        break
    x = todo("apply the layer to x")
    n_layers = todo("one more layer fitted")
    print(f"after layer {n_layers}: output {tuple(x.shape[-2:])}")
print("valid 5x5 layers that fit:", n_layers)

Now try the same stack with different settings, 20 layers deep, and see
which ones keep the image at 28×28: `padding=2` (PyTorch's `"same"`
padding for a 5×5 kernel), more filters, a 1×1 kernel, and stride 2.

In [ ]:
def out_size(depth=20, **kwargs):
    x = torch.zeros(1, 1, 28, 28)
    for _ in range(depth):
        if min(x.shape[-2:]) < kwargs.get("kernel_size", 5):
            return "empty (the kernel no longer fits)"
        x = nn.Conv2d(x.shape[1], kwargs.get("out_channels", 8), kernel_size=kwargs.get("kernel_size", 5),
                      stride=kwargs.get("stride", 1), padding=kwargs.get("padding", 0))(x)
    return tuple(x.shape[-2:])

for settings in [dict(), dict(padding=2), dict(padding="same"), dict(out_channels=64),
                 dict(kernel_size=1), dict(stride=2)]:
    print(f"{str(settings):24s} -> after 20 layers: {out_size(**settings)}")

## 4. A VGG block, and digits in unexpected places

A **VGG block**: two convolutions with "same" padding and ReLU (8 filters
of 5×5, then 8 of 3×3), followed by 2×2 max pooling, which halves each
side of the image.

To test translation, each 28×28 digit is pasted into an empty 56×56
canvas with `place`: either always in the **bottom-right** corner, or at a
**random** position. The training set gets the bottom-right placement,
the test set random placement. The network therefore sees test digits in
places it never saw during training.

In [ ]:
def vgg_block(c_in, c_out=8):
    return todo('two "same" convolutions (c_out filters of 5x5, then c_out of 3x3), each followed by ReLU, then 2x2 max pooling')

def vgg_net(n_blocks, size=56):
    blocks = [vgg_block(1 if i == 0 else 8) for i in range(n_blocks)]
    side = size // 2 ** n_blocks
    return nn.Sequential(*blocks, nn.Flatten(), nn.Linear(8 * side * side, 10))

print("one VGG block on a 56x56 image ->", tuple(vgg_block(1)(torch.zeros(1, 1, 56, 56)).shape))  # (1, 8, 28, 28)

def place(X, random, seed=SEED):
    '''Paste each 28x28 image into a 56x56 canvas (background = standardized 0),
    bottom-right if random=False, at a random offset if random=True.'''
    g = np.random.RandomState(seed)
    out = torch.full((len(X), 1, 56, 56), float((0 - mu) / sd))
    for i in range(len(X)):
        r, c = (g.randint(0, 29), g.randint(0, 29)) if random else (28, 28)
        out[i, :, r:r + 28, c:c + 28] = X[i]
    return out

Xtr_corner, Xtr_random = place(X_train, random=False), place(X_train, random=True, seed=SEED + 1)
Xva_random, Xte_random = place(X_val, random=True, seed=SEED + 2), place(X_test, random=True, seed=SEED + 3)

fig, ax = plt.subplots(1, 2, figsize=(5, 2.5))
ax[0].imshow(Xtr_corner[0, 0], cmap="gray"); ax[0].set_title("training: corner")
ax[1].imshow(Xte_random[0, 0], cmap="gray"); ax[1].set_title("test: random")
for a in ax: a.axis("off")
plt.show()

In [ ]:
corner_results = {}
for n_blocks in [1, 3]:
    torch.manual_seed(SEED)
    m = train(vgg_net(n_blocks), Xtr_corner, y_train, epochs=5)
    corner_results[n_blocks] = (accuracy(m, place(X_val, random=False), y_val), accuracy(m, Xva_random, y_val))
    print(f"{n_blocks} VGG block(s), trained on corner digits: validation accuracy "
          f"corner {corner_results[n_blocks][0]:.3f}, random position {corner_results[n_blocks][1]:.3f}")

Now **augment**: train the 3-block network on randomly placed digits
instead, and test again on randomly placed validation digits.

In [ ]:
torch.manual_seed(SEED)
vgg3 = train(vgg_net(3), Xtr_random, y_train, epochs=5)
acc_vgg_random = todo('validation accuracy of vgg3 on the randomly placed validation digits')
print(f"3 VGG blocks trained on random positions: validation accuracy {acc_vgg_random:.3f}")

## 5. A ResNet block

A **ResNet block** adds a *skip connection*: the block's input goes
around the convolutions and is added to their output, so the block only
has to learn a correction to its input. Here:

- main path: Conv 8 filters 5×5 "same" + ReLU, then Conv 8 filters 5×5
  "same" (no activation);
- skip path: Conv 8 filters 1×1 (to match the number of channels);
- output: ReLU(main + skip).

Three blocks, each followed by 2×2 max pooling, then `Flatten` and a
`Linear` layer to the 10 outputs; trained on randomly placed digits like
`vgg3`.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c_in, c_out=8):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size=5, padding="same")
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size=5, padding="same")
        self.skip = nn.Conv2d(c_in, c_out, kernel_size=1)

    def forward(self, x):
        main = self.conv2(torch.relu(self.conv1(x)))
        return todo("ReLU of (main path + skip path applied to x)")

resnet = nn.Sequential(ResBlock(1), nn.MaxPool2d(2), ResBlock(8), nn.MaxPool2d(2), ResBlock(8), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(8 * 7 * 7, 10))
print("VGG (3 blocks):", count(vgg_net(3)), "parameters;  ResNet (3 blocks):", count(resnet), "parameters")
torch.manual_seed(SEED)
resnet = train(resnet, Xtr_random, y_train, epochs=5)
acc_res_random = accuracy(resnet, Xva_random, y_val)
print(f"validation accuracy on random positions: VGG {acc_vgg_random:.3f}   ResNet {acc_res_random:.3f}")

## 6. Rotation: not built in

Convolution shares weights across *positions*, so a shifted digit gives a
shifted feature map (Day 10 page). Nothing similar holds for *rotation*.
Train a small CNN on the ordinary 28×28 training images, then test it on
validation digits rotated by 45°. Then train the same network again with
**rotation augmentation**: every training batch is rotated by a random
angle between −45° and +45°.

In [ ]:
def small_cnn():
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(32 * 7 * 7, 10))

background = float((0 - mu) / sd)
rotate = lambda X, angle: TF.rotate(X, angle, fill=background)

def random_rotation(xb):
    angles = torch.empty(len(xb)).uniform_(-45, 45)
    return torch.stack([rotate(x, float(a)) for x, a in zip(xb, angles)])

X_val_rot45 = rotate(X_val, 45.0)
rotation_results = {}
for name, augment in [("no augmentation", None), ("rotation augmentation", random_rotation)]:
    torch.manual_seed(SEED)
    m = train(small_cnn(), X_train, y_train, epochs=5, augment=augment)
    rotation_results[name] = todo('a tuple (accuracy on X_val, accuracy on X_val_rot45)')
    print(f"{name:22s}: validation accuracy upright {rotation_results[name][0]:.3f}, rotated 45 deg {rotation_results[name][1]:.3f}")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.